In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import argparse
import scanpy as sc
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import sys
sys.path.append("..") 

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter
import anndata as ad
import pandas as pd
from collections import Counter
import anndata
from sklearn.model_selection import train_test_split

from lib_vqvae.vqvae import VQVAE
from lib_vqvae.dataset import *

#### Load the pre-trained cell tokenizer

In [2]:
import warnings
warnings.filterwarnings("ignore")
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("--train_batch_size", type=int, default=64)
parser.add_argument("--eval_batch_size", type=int, default=2000)
parser.add_argument("--encoder_hidden_dim", type=list, default=[1600,1024,800])
parser.add_argument("--decoder_hidden_dim", type=list, default=[800,1024,1600])
parser.add_argument("--codebook_dim", type=int, default=8)
parser.add_argument("--n", type=int, default=32)
parser.add_argument("--n_codebooks", type=int, default=256)
parser.add_argument("--gamma", type=float, default=5)
parser.add_argument("--beta", type=float, default=10)
parser.add_argument('--weight_decay', type=float, default=5e-4)
parser.add_argument("--lr", type=float, default=1e-4) # 1e-3
parser.add_argument('--max_iter', type=int, default=20)
parser.add_argument('--seed', type=int, default=2024)
parser.add_argument('--gpu', type=int, default=0)
parser.add_argument("--dropout_rate", type=float, default=0.2)
parser.add_argument('--mode', type=str, default="vqvae")
parser.add_argument('--hvg', type=int, default=5000)

args = parser.parse_args([])

# -- device
if torch.cuda.is_available():
    args.device='cuda'
    torch.cuda.set_device(args.gpu)
else:
    args.device='cpu' 
print(args.device)
    
data_dim = 5000
vae = VQVAE(data_dim, args.n, args.codebook_dim, args.n_codebooks, args.encoder_hidden_dim, args.decoder_hidden_dim, args.beta, args.gamma, args.dropout_rate).to(args.device)

vae.load_state_dict(torch.load('../model/cell_tokenzier/model.pth')['model'])
vae.eval() 

cuda


VQVAE(
  (encoder): Encoder(
    (network): Sequential(
      (0): Linear(in_features=5000, out_features=1600, bias=True)
      (1): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
      (2): PReLU(num_parameters=1)
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=1600, out_features=1024, bias=True)
      (5): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (6): PReLU(num_parameters=1)
      (7): Dropout(p=0.2, inplace=False)
      (8): Linear(in_features=1024, out_features=800, bias=True)
      (9): LayerNorm((800,), eps=1e-05, elementwise_affine=True)
      (10): PReLU(num_parameters=1)
      (11): Dropout(p=0.2, inplace=False)
      (12): Linear(in_features=800, out_features=256, bias=True)
    )
  )
  (vector_quantization): VectorQuantizer(
    (embedding): Embedding(256, 8)
  )
  (decoder): Decoder(
    (network): Sequential(
      (0): Linear(in_features=256, out_features=800, bias=True)
      (1): LayerNorm((800,), eps=1e-05, elementwise

#### Load your code data got from tutorial 1

In [3]:
import scanpy as sc
import anndata as ad

adata_code = sc.read_h5ad("../data/blood/blood_code.h5ad")
adata_code

AnnData object with n_obs × n_vars = 4755 × 5000
    obs: 'donor_id', 'author_cell_type', 'days_post_intubation', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'organism_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'batch'
    var: 'feature_name'
    uns: 'log1p'
    obsm: 'code_index'

#### Change this adata.obs into df

In [4]:
import pandas as pd

df = adata_code.obs.copy()
df

,donor_id,author_cell_type,days_post_intubation,assay_ontology_term_id,cell_type_ontology_term_id,development_stage_ontology_term_id,disease_ontology_term_id,self_reported_ethnicity_ontology_term_id,is_primary_data,organism_ontology_term_id,...,suspension_type,cell_type,assay,disease,organism,sex,tissue,self_reported_ethnicity,development_stage,batch
index,,,,,,,,,,,,,,,,,,,,,
AAACCCACAACACACT,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,cell,mature NK T cell,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch
AAACCCACAAGTTTGC,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,cell,mature NK T cell,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch
AAACCCACAGAACCGA,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,cell,mature NK T cell,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch
AAACCCAGTACTGTTG,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,cell,mature NK T cell,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch
AAACCCAGTATACAGA,COV026,myeloid,8,EFO:0009922,CL:0000763,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,cell,myeloid cell,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGTTTCCTTATAC,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,cell,mature NK T cell,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch
TTTGTTGGTGCTGCAC,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,cell,mature NK T cell,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch
TTTGTTGGTGGCGCTT,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,cell,mature NK T cell,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch


#### Then you can construct stories as you like using these metadata
Here, we provide a simplified version of the story construction method.

In [5]:
def generate_story(row):
    components = {
        'cell_type': row['cell_type'] or 'unknown cell type',
        'tissue': row['tissue'] or 'unknown tissue',
        'development_stage': row['development_stage'] or 'unknown stage',
        'assay': row['assay'] or 'unknown assay',
        'disease': row['disease'] or 'no disease',
        'ethnicity': row['self_reported_ethnicity'] or 'unknown',
        'sex': row['sex'].lower() if pd.notna(row['sex']) else 'unknown sex'
    }

    story = (
        f"This cell is a {components['cell_type']} from the {components['tissue']} "
        f"of a {components['sex']} in the {components['development_stage']}, "
        f"analyzed using the {components['assay']}, and associated with {components['disease']}. "
        f"The self-reported ethnicity is {components['ethnicity']}."
    )
    return story

df['story'] = df.apply(generate_story, axis=1)

df

,donor_id,author_cell_type,days_post_intubation,assay_ontology_term_id,cell_type_ontology_term_id,development_stage_ontology_term_id,disease_ontology_term_id,self_reported_ethnicity_ontology_term_id,is_primary_data,organism_ontology_term_id,...,cell_type,assay,disease,organism,sex,tissue,self_reported_ethnicity,development_stage,batch,story
index,,,,,,,,,,,,,,,,,,,,,
AAACCCACAACACACT,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,mature NK T cell,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch,This cell is a mature NK T cell from the blood...
AAACCCACAAGTTTGC,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,mature NK T cell,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch,This cell is a mature NK T cell from the blood...
AAACCCACAGAACCGA,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,mature NK T cell,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch,This cell is a mature NK T cell from the blood...
AAACCCAGTACTGTTG,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,mature NK T cell,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch,This cell is a mature NK T cell from the blood...
AAACCCAGTATACAGA,COV026,myeloid,8,EFO:0009922,CL:0000763,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,myeloid cell,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch,This cell is a myeloid cell from the blood of ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGTTTCCTTATAC,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,mature NK T cell,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch,This cell is a mature NK T cell from the blood...
TTTGTTGGTGCTGCAC,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,mature NK T cell,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch,This cell is a mature NK T cell from the blood...
TTTGTTGGTGGCGCTT,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,mature NK T cell,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch,This cell is a mature NK T cell from the blood...


In [6]:
df['story'][1660]

"This cell is a mature NK T cell from the blood of a female in the 74-year-old human stage, analyzed using the 10x 3' v3, and associated with COVID-19. The self-reported ethnicity is unknown."

Add code information

In [7]:
code_index_array = np.array(adata_code.obsm["code_index"])

df['code_index_list'] = [code_index_array[adata_code.obs.index.get_loc(cell)] for cell in df.index]
df

,donor_id,author_cell_type,days_post_intubation,assay_ontology_term_id,cell_type_ontology_term_id,development_stage_ontology_term_id,disease_ontology_term_id,self_reported_ethnicity_ontology_term_id,is_primary_data,organism_ontology_term_id,...,assay,disease,organism,sex,tissue,self_reported_ethnicity,development_stage,batch,story,code_index_list
index,,,,,,,,,,,,,,,,,,,,,
AAACCCACAACACACT,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch,This cell is a mature NK T cell from the blood...,"[66, 153, 167, 90, 18, 9, 24, 89, 35, 166, 82,..."
AAACCCACAAGTTTGC,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch,This cell is a mature NK T cell from the blood...,"[216, 66, 50, 129, 150, 3, 29, 118, 254, 243, ..."
AAACCCACAGAACCGA,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch,This cell is a mature NK T cell from the blood...,"[63, 212, 75, 241, 214, 202, 120, 111, 20, 13,..."
AAACCCAGTACTGTTG,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch,This cell is a mature NK T cell from the blood...,"[38, 15, 65, 22, 235, 103, 170, 65, 213, 124, ..."
AAACCCAGTATACAGA,COV026,myeloid,8,EFO:0009922,CL:0000763,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch,This cell is a myeloid cell from the blood of ...,"[159, 1, 67, 208, 16, 35, 66, 219, 196, 71, 75..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGTTTCCTTATAC,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch,This cell is a mature NK T cell from the blood...,"[227, 144, 219, 214, 220, 32, 20, 65, 100, 124..."
TTTGTTGGTGCTGCAC,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch,This cell is a mature NK T cell from the blood...,"[178, 5, 219, 52, 129, 137, 20, 65, 100, 234, ..."
TTTGTTGGTGGCGCTT,COV026,Tcell/NK,8,EFO:0009922,CL:0000814,HsapDv:0000168,MONDO:0100096,unknown,True,NCBITaxon:9606,...,10x 3' v3,COVID-19,Homo sapiens,female,blood,unknown,74-year-old human stage,batch,This cell is a mature NK T cell from the blood...,"[136, 2, 186, 46, 101, 91, 211, 118, 106, 59, ..."


#### Construct Huggingface dataset

In [8]:
def create_hf_format(row):
    codes = row['code_index_list'] 
    
    code_tokens = [f'<code_{code}>' for code in codes]
    
    input_text = '<cell_token_start>' + ''.join(code_tokens) + '<cell_token_end>, describe this cell: '
    
    output_text = row['story']
    
    return {
        'input_text': input_text,
        'output_text': output_text
    }

In [9]:
hf_data = [
    create_hf_format(row) for _, row in df.iterrows()
]

In [10]:
from datasets import Dataset

hf_dataset = Dataset.from_list(hf_data)
hf_dataset

Dataset({
    features: ['input_text', 'output_text'],
    num_rows: 4755
})

#### Tokenization

In [28]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer)

# load the tokenizer of Qwen
tokenizer = AutoTokenizer.from_pretrained("your path to the LLM's tokenizer/")

In [29]:
def preprocess_function_tokenized(examples):
    batch_size = len(examples[text_column])
    targets = [str(x) for x in examples[label_column]]
    model_inputs = tokenizer(examples[text_column])
    labels = tokenizer(targets, add_special_tokens=False)  # don't add bos token because we concatenate with inputs
    for i in range(batch_size):
        sample_input_ids = model_inputs["input_ids"][i]
        label_input_ids = labels["input_ids"][i] + [tokenizer.eos_token_id]
        # print(i, sample_input_ids, label_input_ids)
        model_inputs["input_ids"][i] = sample_input_ids + label_input_ids
        labels["input_ids"][i] = [-100] * len(sample_input_ids) + label_input_ids
        model_inputs["attention_mask"][i] = [1] * len(model_inputs["input_ids"][i])
#     print(model_inputs)
    for i in range(batch_size):
        sample_input_ids = model_inputs["input_ids"][i]
        label_input_ids = labels["input_ids"][i]
        model_inputs["input_ids"][i] = [tokenizer.pad_token_id] * (
            max_length - len(sample_input_ids)
        ) + sample_input_ids
        model_inputs["attention_mask"][i] = [0] * (max_length - len(sample_input_ids)) + model_inputs[
            "attention_mask"
        ][i]
        labels["input_ids"][i] = [-100] * (max_length - len(sample_input_ids)) + label_input_ids
        model_inputs["input_ids"][i] = model_inputs["input_ids"][i][:max_length]
        model_inputs["attention_mask"][i] = model_inputs["attention_mask"][i][:max_length]
        labels["input_ids"][i] = labels["input_ids"][i][:max_length]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

text_column = "input_text"
label_column = "output_text"
max_length = 200
batch_size = 30

In [30]:
processed_datasets = hf_dataset.map(
    preprocess_function_tokenized,
    batched=True,
    num_proc=10,
    load_from_cache_file=True,
    desc="Running tokenizer on dataset",
)
processed_datasets

Running tokenizer on dataset (num_proc=10):   0%|          | 0/4755 [00:00<?, ? examples/s]

Dataset({
    features: ['input_text', 'output_text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 4755
})

In [31]:
processed_datasets.save_to_disk("../data/blood/tokenized_dataset_annotation/", max_shard_size='20GB', num_proc=10)

Saving the dataset (0/10 shards):   0%|          | 0/4755 [00:00<?, ? examples/s]